# StormScope-EU `true_color` forecast verification with `xskillscore`

Scores the nowcasts in `gtc-berlin/stormscope-eu_gtc_berlin_2026_EU_region` against what MTG-I1 FCI
actually observed, per lead time, and against a persistence baseline.

## Run on Coiled

The Arraylake bucket is in `eu-west-1`, so run the notebook VM there (CPU only, no GPU needed).
From the repo root, with `ARRAYLAKE_API_TOKEN` in `.env` (see `.env.example`):

```bash
uv sync --group notebook
set -a; source .env; set +a;
uv run --group notebook coiled notebook start --sync --name $USER-stormscope-verification \
    --workspace earthmover-devs --vm-type 'm6i.2xlarge' --region 'eu-west-1' \
    --open --idle-timeout '30 minutes' --allow-sync-from me \
    --env ARRAYLAKE_API_TOKEN=$ARRAYLAKE_API_TOKEN
```

`--sync` package-syncs the local uv env (incl. the `notebook` group: `xskillscore`, `dask`,
`matplotlib`) to the VM and mirrors the working directory, so this notebook shows up in JupyterLab.
Stop the VM when done: `coiled notebook stop $USER-stormscope-verification --workspace earthmover-devs`.
Do not `coiled run --name` this cluster: it shuts the VM down after the command.

## Method

- Window: daytime inits (06-15 UTC, true colour is black at night) over a Germany lat/lon box.
- `lead_time = 0` is the observed FCI analysis frame of each init, so it is the verification
  truth: a forecast from init `t` at lead `L` is scored against the `lead_time = 0` frame of the
  init at `t + L`. Pairs whose valid time has no init (night, pipeline gaps) are NaN and skipped.
- Persistence baseline: the `lead_time = 0` frame of init `t` used as the forecast for `t + L`.
- ACC: Pearson correlation per (init, lead) over `(x, y, rgb)`, for the model, persistence, and a
  climatology baseline (model forecasts shuffled across inits, so each meets an unrelated observation).
- Change correlation: correlate the forecast change since init with the observed change, so the
  static scene (coasts, terminator, stationary cloud decks) no longer inflates the score.
  Persistence scores 0 by construction.
- Fractions skill score (FSS) of a brightness cloud mask over neighbourhood sizes: tolerant to
  small displacements, gives the scale from which the forecast is useful at each lead.

In [28]:
!pip install seaborn -q

In [1]:
import seaborn as sns

sns.set_palette('viridis',12)

In [2]:
import os

import arraylake
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import xskillscore as xs
from dotenv import load_dotenv
from scipy.ndimage import uniform_filter

load_dotenv()
client = arraylake.Client(token=os.environ.get("ARRAYLAKE_API_TOKEN"))  # None: use `arraylake auth login` credentials
repo = client.get_repo("gtc-berlin/stormscope-eu_gtc_berlin_2026_EU_region")
session = repo.readonly_session("main")
#sel_time = slice("2026-10-15", "2026-10-20")
ds = xr.open_zarr(session.store, chunks={}, consolidated=False).squeeze()["true_color"]
sel_hours = ds.time.dt.hour.isin([6,7,8,9,10,11,12,13,14,15]) # day time
ds = ds.sel(time=sel_hours)

In [3]:
inside = ((ds.lat > 47.3) & (ds.lat < 55.1) & (ds.lon > 5.9) & (ds.lon < 15.0)).compute() # germany
ds = ds.isel(y=inside.any("x"), x=inside.any("y")).chunk({'x':-1,'y':-1})

In [4]:
fc = ds.drop_isel(lead_time=0)
obs = ds.isel(lead_time=0)

## Observations at valid time

`obs` is indexed by init time. The truth for (init `t`, lead `L`) is the lead-0 frame of the init
at `t + L`, so look it up via `valid_time`. `obs` itself, the frame at init, is the persistence forecast.

In [ ]:
dims = ["x", "y", "rgb"]
fc32 = fc.astype("float32").drop_vars("valid_time")  # float: NaN for missing pairs
pers = obs.drop_vars(["valid_time", "lead_time"]).astype("float32")  # frame at init = persistence forecast
vt = fc.valid_time.compute()
obs_vt = (
    pers.reindex(time=np.unique(vt.values))  # NaN where no init exists at the valid time
    .sel(time=vt.variable)  # obs at time + lead_time, same shape as fc
    .assign_coords(time=fc.time, lead_time=fc.lead_time)
)

## ACC: model vs persistence

In [ ]:
acc = xr.Dataset({"model": xs.pearson_r(fc32, obs_vt, dim=dims, skipna=True), "persistence": xs.pearson_r(pers, obs_vt, dim=dims, skipna=True)}).compute()

## Climatology baseline by shuffling forecasts

Pair each forecast with the observation of a random other init (same lead), averaged over
`N_SHUFFLE` shuffles. It keeps the forecast's own colour/cloud distribution but removes timing.
Each shuffle reads the data again.

In [ ]:
N_SHUFFLE = 3
rng = np.random.default_rng(0)
nt = fc32.sizes["time"]
shuffled = [fc32.isel(time=rng.permutation(nt)).assign_coords(time=fc32.time) for _ in range(N_SHUFFLE)]
acc["climatology"] = xr.concat([xs.pearson_r(s, obs_vt, dim=dims, skipna=True) for s in shuffled], "iteration").mean("iteration").compute()

In [ ]:
acc.mean("time").to_dataarray("forecast").plot(hue="forecast", x="lead_time", marker="o")
plt.title("ACC true_color, Germany")

In [ ]:
acc.model.plot(hue="lead_time", x="time")

## Correlation of changes since init

Spatial correlation of raw frames is dominated by what does not move. Correlating the change
`forecast - obs_at_init` with `obs - obs_at_init` asks only whether the model predicted what changed.

In [ ]:
r_change = xs.pearson_r(fc32 - pers, obs_vt - pers, dim=dims, skipna=True).compute()
r_change.mean("time").plot(x="lead_time", marker="o")
plt.title("change correlation true_color, Germany")

## FSS: model vs persistence

Cloud mask = band-mean brightness > `THRESHOLD`. Fractions are the cloudy share in an `n x n`
pixel window. Errors are summed over all inits before the ratio (Roberts & Lean 2008); pairs
without an observed frame at valid time are dropped. Useful skill: FSS > 0.5 + f/2, f = observed cloud fraction.

In [ ]:
THRESHOLD = 150  # 0-255 band-mean brightness; cloud above
SCALES = [1, 5, 11, 25, 51, 101]  # window size in pixels, ~2 km each


def fractions(binary, n):
    return xr.apply_ufunc(
        lambda a: uniform_filter(a, size=(1,) * (a.ndim - 2) + (n, n), mode="constant"),
        binary.astype("float32"),
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        dask="parallelized",
        output_dtypes=[np.float32],
    )


bright_obs = obs_vt.mean("rgb")
valid = bright_obs.notnull()  # drop pairs without an observed frame at valid time
fss = []
for n in SCALES:
    po = fractions(bright_obs > THRESHOLD, n)
    for name, forecast in (("model", fc32), ("persistence", pers)):
        pf = fractions(forecast.mean("rgb") > THRESHOLD, n)
        num = ((pf - po) ** 2).where(valid).sum(["time", "y", "x"])
        den = (pf**2 + po**2).where(valid).sum(["time", "y", "x"])
        fss.append((1 - num / den).assign_coords(scale_km=2 * n, forecast=name))
fss = xr.concat(fss, "stack").set_index(stack=["scale_km", "forecast"]).unstack("stack").compute()
f = float((bright_obs > THRESHOLD).where(valid).mean())
print(f"observed cloud fraction {f:.2f}, useful FSS > {0.5 + f / 2:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, name in zip(axes, ("model", "persistence"), strict=True):
    fss.sel(forecast=name).plot(ax=ax, hue="scale_km", x="lead_time", marker="o")
    ax.axhline(0.5 + f / 2, color="k", ls="--", lw=1)
    ax.set_title(f"FSS {name}, Germany")